# Préprocessing des données marketing

## Objectif
Préparer un dataset propre et exploitable pour la phase de modélisation.

Dans ce notebook, on va :
1. charger les données brutes ;
2. appliquer le nettoyage de base ;
3. créer la variable cible `ROI` ;
4. traiter les valeurs extrêmes du ROI ;
5. encoder la variable catégorielle `Influencer` ;
6. séparer les features `X` et la target `y` ;
7. créer les jeux d'entraînement et de test.

## Rappel méthodologique
Le ROI est défini ici comme :

`ROI = Sales / (TV + Radio + Social Media)`

Attention : on retire `Sales` des variables explicatives pour éviter toute fuite de données (*data leakage*).


In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mon_projet.data.load_data import load_raw_data
from mon_projet.data.preprocess import (
    clean_data,
    create_roi,
    handle_outliers,
    encode_features,
    prepare_features_target,
    split_data
)

pd.set_option("display.max_columns", None)


## 1. Chargement des données brutes
On part du fichier source présent dans `data/raw/`.


In [ ]:
data_path = "../data/raw/marketing_and_sales.xls"
df_raw = load_raw_data(data_path)

df_raw.head()


In [ ]:
print("Dimensions initiales :", df_raw.shape)
print("\nColonnes :")
print(df_raw.columns.tolist())


## 2. Nettoyage de base
On applique le nettoyage défini dans `preprocess.py` :
- suppression des doublons ;
- homogénéisation légère des noms de colonnes ;
- imputation simple des valeurs manquantes numériques par la médiane.


In [ ]:
df = clean_data(df_raw)

print("Dimensions après nettoyage :", df.shape)
df.head()


In [ ]:
print("Valeurs manquantes après nettoyage :")
display(df.isna().sum())


## 3. Création de la cible ROI
On construit maintenant la variable cible du projet.


In [ ]:
df = create_roi(df)

df[["TV", "Radio", "Social Media", "Sales", "ROI"]].head()


In [ ]:
df["ROI"].describe()


### Lecture attendue
À ce stade, le ROI peut présenter une distribution très asymétrique avec quelques valeurs extrêmes.
C'est normal, car il s'agit d'un ratio sensible à de petits budgets.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["ROI"], bins=40, kde=True)
plt.title("Distribution du ROI avant traitement des outliers")
plt.xlabel("ROI")
plt.show()


## 4. Traitement des valeurs extrêmes
On applique ici une stratégie simple de *capping* du ROI afin de limiter l'effet des valeurs extrêmes sur les futurs modèles.


In [ ]:
df = handle_outliers(df)

df["ROI"].describe()


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["ROI"], bins=40, kde=True)
plt.title("Distribution du ROI après traitement des outliers")
plt.xlabel("ROI")
plt.show()


### Interprétation
Après ce traitement, la cible devient plus stable pour les algorithmes de régression.
On conserve l'information utile tout en réduisant l'influence disproportionnée des observations extrêmes.


## 5. Encodage de la variable catégorielle
Les modèles de machine learning ne peuvent pas travailler directement avec du texte.
On encode donc `Influencer` en variables binaires via un *one-hot encoding*.


In [ ]:
print("Valeurs uniques de Influencer avant encodage :")
display(df["Influencer"].value_counts(dropna=False))


In [ ]:
df_encoded = encode_features(df)

df_encoded.head()


In [ ]:
print("Colonnes après encodage :")
print(df_encoded.columns.tolist())


## 6. Préparation de X et y
On sépare maintenant :
- `y` : la cible `ROI`
- `X` : les variables explicatives

On retire volontairement `Sales` pour éviter une fuite de données, car `Sales` sert déjà au calcul du ROI.


In [ ]:
X, y = prepare_features_target(df_encoded)

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)


In [ ]:
display(X.head())
display(y.head())


## 7. Découpage train / test
On découpe les données pour préparer la phase de modélisation.


In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print("Train X :", X_train.shape)
print("Test X  :", X_test.shape)
print("Train y :", y_train.shape)
print("Test y  :", y_test.shape)


## 8. Vérifications finales
On vérifie rapidement qu'il n'y a pas de problème structurel dans les jeux produits.


In [ ]:
print("Valeurs manquantes dans X_train :")
display(X_train.isna().sum())

print("\nValeurs manquantes dans X_test :")
display(X_test.isna().sum())


In [ ]:
print("Types de données de X_train :")
display(X_train.dtypes)


## 9. Sauvegarde intermédiaire optionnelle
Cette étape permet de conserver une version prétraitée du dataset pour la suite du projet.


In [ ]:
output_path = "../data/processed/marketing_and_sales_preprocessed.csv"
df_encoded.to_csv(output_path, index=False)

print(f"Fichier sauvegardé : {output_path}")


## Conclusion
Le dataset est désormais prêt pour la phase de modélisation.

### Résultat de cette étape
- données nettoyées ;
- variable cible `ROI` créée ;
- outliers du ROI traités ;
- variable `Influencer` encodée ;
- séparation `X / y` réalisée ;
- jeux `train / test` prêts.

### Étape suivante
Construire un premier modèle de référence (*baseline*), par exemple une régression linéaire.
